### 下载qwen2.5-3B-instruct

In [1]:
import tqdm as notebook_tqdm
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct",
    local_dir="models/Qwen2.5-3B-Instruct",
    local_dir_use_symlinks=False
)
print("done")

/opt/anaconda3/envs/py38/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/py38/lib/python3.8/site-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 4314.76it/s]

done


## 进行小参数LLM微调

### 基础配置

In [2]:
from pathlib import Path

# 本地模型目录
MODEL_DIR = Path("models/Qwen2.5-3B-Instruct")

# 前面生成的 prompt 数据文件
PROMPT_FILE = Path("outputs/prompts/recipe_success.jsonl")

# 先做 success 任务；跑通后改成 "topology"
TASK = "success"

# 输出目录
OUTPUT_DIR = Path(f"outputs/lora_qwen25_3b_{TASK}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(MODEL_DIR.resolve())
print(PROMPT_FILE.resolve())
print(OUTPUT_DIR.resolve())

/Users/tangren/Documents/ZeoScreen/models/Qwen2.5-3B-Instruct
/Users/tangren/Documents/ZeoScreen/outputs/prompts/recipe_success.jsonl
/Users/tangren/Documents/ZeoScreen/outputs/lora_qwen25_3b_success


### 读取prompt命令，只保留当前任务（success）

In [3]:
import json
import pandas as pd

rows = []
with open(PROMPT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df = df[df["task"] == TASK].copy().reset_index(drop=True)

print(df.shape)
display(df.head(3))

(30164, 9)


,task,route_id,paper_id,doi,group_id,instruction,input,output,target_failure_label
0,success,1,12.0,10.1002/1521-3765(20021004)8:19<4549::aid-chem...,10.1002/1521-3765(20021004)8:19<4549::aid-chem...,Predict whether the zeolite synthesis will be ...,Zeolite hydrothermal synthesis recipe.\nSilica...,success,success
1,success,2,12.0,10.1002/1521-3765(20021004)8:19<4549::aid-chem...,10.1002/1521-3765(20021004)8:19<4549::aid-chem...,Predict whether the zeolite synthesis will be ...,Zeolite hydrothermal synthesis recipe.\nSilica...,success,success
2,success,3,12.0,10.1002/1521-3765(20021004)8:19<4549::aid-chem...,10.1002/1521-3765(20021004)8:19<4549::aid-chem...,Predict whether the zeolite synthesis will be ...,Zeolite hydrothermal synthesis recipe.\nSilica...,success,success


### 检查标签分布

In [4]:
print(df["output"].value_counts(dropna=False).head(30))

output
failed     16286
success    13878
Name: count, dtype: int64


### 按 group_id 切分 train / valid / test

In [5]:
from sklearn.model_selection import GroupShuffleSplit

# 先切出 test
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_valid_idx, test_idx = next(gss1.split(df, groups=df["group_id"]))

train_valid_df = df.iloc[train_valid_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# 再从 train_valid 中切出 valid
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)  
# 0.1765 * 0.85 ≈ 0.15
train_idx, valid_idx = next(gss2.split(train_valid_df, groups=train_valid_df["group_id"]))

train_df = train_valid_df.iloc[train_idx].reset_index(drop=True)
valid_df = train_valid_df.iloc[valid_idx].reset_index(drop=True)

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

print("train groups:", train_df["group_id"].nunique())
print("valid groups:", valid_df["group_id"].nunique())
print("test  groups:", test_df["group_id"].nunique())

train: (21489, 9)
valid: (4914, 9)
test : (3761, 9)
train groups: 6525
valid groups: 1399
test  groups: 1399


### 转成 prompt-completion 数据格式

In [6]:
def build_prompt(row):
    return (
        f"Instruction: {row['instruction']}\n\n"
        f"Input:\n{row['input']}\n\n"
        f"Output:\n"
    )

train_ds_df = pd.DataFrame({
    "prompt": train_df.apply(build_prompt, axis=1),
    "completion": train_df["output"].astype(str),
})

valid_ds_df = pd.DataFrame({
    "prompt": valid_df.apply(build_prompt, axis=1),
    "completion": valid_df["output"].astype(str),
})

test_ds_df = pd.DataFrame({
    "prompt": test_df.apply(build_prompt, axis=1),
    "completion": test_df["output"].astype(str),
})

display(train_ds_df.head(2))

,prompt,completion
0,Instruction: Predict whether the zeolite synth...,success
1,Instruction: Predict whether the zeolite synth...,success


### 转成 Hugging Face Dataset

In [7]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_ds_df, preserve_index=False)
valid_ds = Dataset.from_pandas(valid_ds_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_ds_df, preserve_index=False)

train_ds, valid_ds, test_ds

(Dataset({
     features: ['prompt', 'completion'],
     num_rows: 21489
 }),
 Dataset({
     features: ['prompt', 'completion'],
     num_rows: 4914
 }),
 Dataset({
     features: ['prompt', 'completion'],
     num_rows: 3761
 }))

### 加载tokenizer模型

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)

# Qwen 系列常见处理：如果没有 pad_token，就用 eos_token 代替
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 自动选择 dtype
if torch.cuda.is_available():
    bf16_ok = torch.cuda.is_bf16_supported()
    model_dtype = torch.bfloat16 if bf16_ok else torch.float16
else:
    model_dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=model_dtype,
    device_map="mps",
)
model.config.use_cache = False

print("dtype:", model_dtype)
print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.62s/it]

dtype: torch.float32
pad_token: <|endoftext|>
eos_token: <|im_end|>


### 配置LoRA

In [9]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

peft_config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type='CAUSAL_LM', inference_mode=False, r=16, target_modules={'o_proj', 'k_proj', 'q_proj', 'v_proj'}, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))

### 配置 SFTTrainer

In [10]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,          # LoRA 常用较高学习率
    num_train_epochs=3,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and (not torch.cuda.is_bf16_supported()),
    gradient_checkpointing=False,
    dataset_num_proc=1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    peft_config=peft_config,
    max_seq_length=512,
)

trainer

/opt/anaconda3/envs/py38/lib/python3.8/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/opt/anaconda3/envs/py38/lib/python3.8/site-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
Map: 100%|██████████| 4914/4914 [00:00<00:00, 12977.32 examples/s]
/opt/anaconda3/envs/py38/lib/python3.8/site-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


### 开始训练

In [ ]:
trainer.train()

  0%|          | 2/8058 [00:39<47:35:18, 21.27s/it]